<div class="alert alert-block alert-success">

# App 1: 多模态模型训练 (Multimodal Model Training)

**项目:** FIT5196 A1 (Extended Part)  
**模块:** App 1 - 评分预测模型 (Rating Prediction)  
**架构:** RoBERTa-Large + Swin-Base (with LoRA)  
**日期:** 2025.11.21

</div>

**概览 (Overview):**
本 Notebook 实现了 App 1 的核心训练流程。我们构建了一个 **双塔多模态架构 (Two-Tower Multimodal Architecture)**，利用文本和图像特征共同预测用户评分。

为了在有限的计算资源下实现高性能，本方案集成了多项前沿技术：**LoRA (低秩微调)** 以降低显存需求，**混合精度训练 (AMP)** 加速计算，以及 **Focal Loss** 解决数据不平衡问题。

<div class="alert alert-block alert-info">

## 训练管道架构 (Training Pipeline)

本 Notebook 的执行流程设计如下：

1.  **Colab I/O 优化:** 针对 Google Drive 读取速度慢的瓶颈，实施“云端存储 -> 虚拟机本地解压”策略，将 I/O 速度提升百倍。
2.  **数据增强 (针对小图):** 针对数据集中图片偏小 (约 150px) 的特点，定制了基于 **BICUBIC 插值** 的增强策略，防止放大后失真。
3.  **模型构建 (LoRA):**     
    * **文本塔:** RoBERTa-Large (注入 LoRA 适配器)
    * **图像塔:** Swin-Base Transformer (注入 LoRA 适配器)
    * **融合层:** Concatenation + MLP
4.  **训练策略:**     
    * **梯度累积 (Gradient Accumulation):** 模拟大 Batch Size (等效 BS=128) 以稳定 BatchNorm。
    * **Mask 机制:** 动态处理缺失模态 (无图样本)。
    * **早停 (Early Stopping):** 监控验证集 F1-Macro 指标。
5.  **评估与交付:** 在独立测试集上计算混淆矩阵，并将最佳模型权重自动备份回 Google Drive。

</div>

## 1. 环境配置与数据搬运 (Environment & Data Setup)
**目标:** 解决 Google Colab 与 Google Drive 之间的 I/O 瓶颈。
* **依赖安装:** 安装 `transformers`, `peft`, `accelerate` 等核心库。
* **极速加载:** 将巨大的 `images.zip` 从 Drive 复制到 Colab 虚拟机的本地 SSD (`/content/images`) 并解压。这比直接从 Drive 读取图片快几个数量级，是训练不卡顿的关键。

In [ ]:
# ==========================================
# 1. 环境配置与数据搬运 (Environment & Data Setup)
# ==========================================

# 1.1 安装依赖库 (如果尚未安装)
# - transformers: HuggingFace 模型库
# - timm: 图像模型库 (虽然主要用HF的Swin，但保留timm以备不时之需)
# - peft: LoRA 微调库
# - accelerate: 混合精度训练支持
try:
    import peft
    import transformers
except ImportError:
    print("正在安装依赖库...")
    !pip install -q transformers peft accelerate timm

import os
import shutil
import torch
from google.colab import drive

# 1.2 挂载 Google Drive
print("正在挂载 Google Drive...")
drive.mount('/content/drive')

# 1.3 定义路径
# 请根据你在 Drive 中的实际文件夹名称修改 DRIVE_BASE_PATH
DRIVE_BASE_PATH = '/content/drive/MyDrive/FIT5196_A1_Extension/App1' 

# Colab 虚拟机本地路径 (速度快)
COLAB_DATA_DIR = '/content/data'
COLAB_IMG_DIR = '/content/images'

# 1.4 数据搬运与解压 (关键步骤)
print("\n正在检查并搬运数据...")

# (1) 搬运 Feather 数据文件
os.makedirs(COLAB_DATA_DIR, exist_ok=True)
for file_name in ['App1_Data_Train.ftr', 'App1_Data_Val.ftr', 'App1_Data_Test.ftr']:
    src = os.path.join(DRIVE_BASE_PATH, file_name)
    dst = os.path.join(COLAB_DATA_DIR, file_name)
    
    if not os.path.exists(dst):
        if os.path.exists(src):
            print(f"正在复制 {file_name} 到本地虚拟机...")
            shutil.copy(src, dst)
        else:
            print(f"警告: 在 Drive 中未找到 {file_name}，请检查路径！")
    else:
        print(f"文件已存在，跳过: {file_name}")

# (2) 搬运并解压图片 (images.zip)
zip_path_drive = os.path.join(DRIVE_BASE_PATH, 'images.zip')
zip_path_colab = '/content/images.zip'

if not os.path.exists(COLAB_IMG_DIR):
    # 检查本地是否已经有解压好的文件夹，如果没有才执行
    if os.path.exists(zip_path_drive):
        if not os.path.exists(zip_path_colab):
            print("正在从 Drive 复制 images.zip (这可能需要几分钟)...")
            shutil.copy(zip_path_drive, zip_path_colab)
        
        print("正在解压 images.zip...")
        # -q: 静默模式, -d: 指定目录
        !unzip -q {zip_path_colab} -d {COLAB_IMG_DIR}
        print("解压完成！")
    else:
        print("错误: 在 Drive 中未找到 images.zip，请确保已上传！")
else:
    print("图片目录已存在，跳过解压。")

print("\n环境准备就绪！")
print(f"数据路径: {COLAB_DATA_DIR}")
print(f"图片路径: {COLAB_IMG_DIR}")

## 2. 全局参数配置 (Configuration)
**目标:** 集中管理所有超参数，确保实验的可复现性。
* **硬件适配:** 针对 A100/T4 GPU 优化 Batch Size 和梯度累积步数。
* **学习率策略:** 采用分层学习率 (Layer-wise LR)，给予预训练层 (LoRA) 较小的学习率，给予分类头 (Head) 较大的学习率。

In [ ]:
# ==========================================
# 2. 全局参数配置 (Configuration)
# ==========================================

class Config:
    # 路径配置
    TRAIN_PATH = f'{COLAB_DATA_DIR}/App1_Data_Train.ftr'
    VAL_PATH = f'{COLAB_DATA_DIR}/App1_Data_Val.ftr'
    # 注意: 解压后的图片可能在 images/ 或 images/03_images_raw/ 下，请根据实际解压结果调整
    # 这里假设 zip 直接解压出图片文件在 COLAB_IMG_DIR 下
    IMAGE_DIR = COLAB_IMG_DIR 
    
    # 模型配置
    TEXT_MODEL = 'roberta-large'          # 文本塔 (HuggingFace ID)
    IMAGE_MODEL = 'microsoft/swin-base-patch4-window7-224' # 图片塔 (HuggingFace ID)
    NUM_CLASSES = 5                       # 1-5星
    MAX_LEN = 256                         # 文本截断长度
    DROPOUT = 0.3
    
    # 训练超参
    EPOCHS = 15
    BATCH_SIZE = 32                       # 物理 Batch Size
    ACCUMULATION_STEPS = 4                # 梯度累积步数 (等效 Batch = 128)
    
    # 学习率 (分层设置)
    LR_TEXT = 1e-4                        # RoBERTa LoRA
    LR_IMG = 1e-4                         # Swin LoRA
    LR_HEAD = 1e-3                        # Classifier Head
    WEIGHT_DECAY = 0.01
    
    # 早停与保存
    PATIENCE = 5
    SEED = 42

# 设置随机种子以保证复现性
def seed_everything(seed):
    import random
    import numpy as np
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(Config.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

## 3. 数据集与增强 (Dataset & Transforms)
**目标:** 构建健壮的数据输入管道。
* **小图优化:** 鉴于原始图片多为 150px 左右的缩略图，强制使用 `BICUBIC` 插值放大至 224px，保留最大细节。
* **多模态对齐:** 实现 `__getitem__` 逻辑，同时返回 tokenized 文本和 tensor 图片。
* **缺失处理:** 对于无图样本，返回全黑张量并设置 `img_mask=0`。

In [ ]:
# ==========================================
# 3. 数据集与增强 (Dataset & Transforms)
# ==========================================

import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoTokenizer

# 针对 150px 小图的增强策略
train_transforms = transforms.Compose([
    # 1. 先用 BICUBIC 高质量放大到 256
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BICUBIC),
    # 2. 随机裁剪 224 (模拟构图微调)
    transforms.RandomCrop(224),
    # 3. 随机水平翻转
    transforms.RandomHorizontalFlip(),
    # 4. 颜色抖动 (亮度/对比度/饱和度)
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    # ImageNet 标准归一化
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    # 验证集直接高质量放大到 224
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class GoogleReviewDataset(Dataset):
    def __init__(self, df_path, img_dir, tokenizer, max_len, is_train=True):
        self.df = pd.read_feather(df_path)
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train
        self.transforms = train_transforms if is_train else val_transforms
        
        # 标签处理: 1-5星 -> 0-4 索引
        self.df['label'] = self.df['review_rating'].astype(int) - 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['review_text'])
        label = row['label']
        img_files = row['img_filenames']

        # 1. 文本处理
        inputs = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        input_ids = inputs['input_ids'].squeeze(0)
        attention_mask = inputs['attention_mask'].squeeze(0)

        # 2. 图片处理
        # 默认全黑图 (全0)，Mask=0
        image_tensor = torch.zeros((3, 224, 224))
        img_mask = torch.tensor(0.0)

        if img_files is not None and len(img_files) > 0:
            if isinstance(img_files, np.ndarray): img_files = img_files.tolist()
            
            # 训练时随机抽一张，验证时抽第一张 (保持确定性)
            target_file = np.random.choice(img_files) if self.is_train else img_files[0]
            img_path = os.path.join(self.img_dir, target_file)
            
            try:
                # 尝试加载图片
                if os.path.exists(img_path):
                    image = Image.open(img_path).convert('RGB')
                    image_tensor = self.transforms(image)
                    img_mask = torch.tensor(1.0) # 标记为有图
            except Exception:
                pass # 加载失败保持全黑

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'image': image_tensor,
            'img_mask': img_mask,
            'label': torch.tensor(label, dtype=torch.long)
        }

## 4. 模型架构 (Model Architecture: RoBERTa + Swin + LoRA)
**目标:** 定义双塔多模态分类器。
* **Text Tower:** `RoBERTa-Large` 用于提取深层语义特征。
* **Image Tower:** `Swin-Base` (Hierarchical Transformer) 用于提取视觉特征。
* **LoRA (Low-Rank Adaptation):** 仅微调注入的低秩矩阵 (约占总参数的 1% 以下)，冻结巨型预训练模型的权重，大幅降低显存占用。
* **Fusion:** 特征拼接后通过 MLP 映射到 5 个评分类别。

In [ ]:
# ==========================================
# 4. 模型架构 (Model Architecture)
# ==========================================

import torch.nn as nn
from transformers import AutoModel, AutoConfig, SwinModel
from peft import get_peft_model, LoraConfig, TaskType

class MultimodalClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        
        # --- 塔 A: 文本 (RoBERTa-Large) ---
        print("正在加载 RoBERTa-Large...")
        self.text_encoder = AutoModel.from_pretrained(Config.TEXT_MODEL)
        
        # 注入 LoRA (针对 Attention 层)
        peft_config_text = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION, 
            r=16, lora_alpha=32, lora_dropout=0.1,
            target_modules=["query", "value"] 
        )
        self.text_encoder = get_peft_model(self.text_encoder, peft_config_text)
        text_dim = self.text_encoder.config.hidden_size # 1024

        # --- 塔 B: 图片 (Swin-Base) ---
        print("正在加载 Swin-Base...")
        self.img_encoder = SwinModel.from_pretrained(Config.IMAGE_MODEL)
        
        # 注入 LoRA (针对 Swin 的 Attention 层)
        # 注意: Swin 的模块命名不同，target_modules 需匹配
        peft_config_img = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=16, lora_alpha=32, lora_dropout=0.1,
            target_modules=["query", "value"] 
        )
        self.img_encoder = get_peft_model(self.img_encoder, peft_config_img)
        img_dim = self.img_encoder.config.hidden_size # 1024
        
        # --- 融合层 (Fusion) ---
        print("初始化融合层...")
        self.fusion_dim = text_dim + img_dim
        self.classifier = nn.Sequential(
            nn.Linear(self.fusion_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, image, img_mask):
        # 1. 文本塔 forward
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        # 取 CLS token 的特征
        text_feat = text_out.last_hidden_state[:, 0, :] 

        # 2. 图片塔 forward
        img_out = self.img_encoder(image)
        # Swin 的输出通常是 [batch, seq_len, dim]，我们取 pooler_output (如果有) 或 mean pooling
        # HuggingFace SwinModel 输出包含 pooler_output [batch, dim]
        img_feat = img_out.pooler_output 
        
        # 3. Mask 机制 (处理无图样本)
        # img_mask: [batch] -> [batch, 1]
        img_mask = img_mask.unsqueeze(1).to(img_feat.device)
        # 如果 mask=0，则图片特征置 0，防止噪音干扰文本特征
        img_feat_masked = img_feat * img_mask
        
        # 4. 拼接
        combined_feat = torch.cat((text_feat, img_feat_masked), dim=1)
        
        # 5. 分类
        logits = self.classifier(combined_feat)
        return logits

    def print_trainable_parameters(self):
        """打印可训练参数量，确认 LoRA 生效"""
        trainable_params = 0
        all_param = 0
        for _, param in self.named_parameters():
            all_param += param.numel()
            if param.requires_grad:
                trainable_params += param.numel()
        print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}")

## 5. 辅助功能 (Utils)
**目标:** 提供训练所需的损失函数和评估工具。
* **Focal Loss:** 专门用于解决类别不平衡问题，增加对“难分类样本” (如 1星、2星) 的权重。
* **Checkpoints:** 实现模型权重的自动保存与云端备份机制。

In [ ]:
# ==========================================
# 5. 辅助功能 (Utils)
# ==========================================

import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# Focal Loss 实现
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean': return focal_loss.mean()
        if self.reduction == 'sum': return focal_loss.sum()
        return focal_loss

# 验证/推理函数
def eval_fn(dataloader, model, criterion, device):
    model.eval()
    total_loss = 0
    preds_all = []
    labels_all = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image = batch['image'].to(device)
            img_mask = batch['img_mask'].to(device)
            labels = batch['label'].to(device)
            
            with torch.cuda.amp.autocast():
                logits = model(input_ids, attention_mask, image, img_mask)
                loss = criterion(logits, labels)
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            preds_all.extend(preds)
            labels_all.extend(labels.cpu().numpy())
            
    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(labels_all, preds_all)
    f1 = f1_score(labels_all, preds_all, average='macro')
    
    return avg_loss, acc, f1

# 模型保存助手
def save_checkpoint(model, filename, drive_folder):
    # 1. 保存到 Colab 本地
    torch.save(model.state_dict(), filename)
    # 2. 立即复制到 Google Drive
    dst = os.path.join(drive_folder, filename)
    shutil.copy(filename, dst)
    print(f"Model saved to Drive: {dst}")

## 6. 训练主循环 (Main Training Loop)
**目标:** 执行标准的深度学习训练流程。
* **混合精度 (AMP):** 使用 `fp16` 计算，减少显存占用并加速训练。
* **梯度累积:** 每 4 步更新一次参数，模拟 Batch Size = 128 的效果。
* **监控:** 实时记录 Train/Val 的 Loss, Accuracy 和 F1-Macro，并实施早停策略 (Patience=5)。

In [ ]:
# ==========================================
# 6. 训练主循环 (Main Training Loop)
# ==========================================

import time
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

def run_training():
    # 1. 准备数据
    print("初始化 Tokenizer & Dataset...")
    tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
    
    train_ds = GoogleReviewDataset(Config.TRAIN_PATH, Config.IMAGE_DIR, tokenizer, Config.MAX_LEN, is_train=True)
    val_ds = GoogleReviewDataset(Config.VAL_PATH, Config.IMAGE_DIR, tokenizer, Config.MAX_LEN, is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    
    # 2. 初始化模型
    model = MultimodalClassifier(num_classes=Config.NUM_CLASSES)
    model.to(device)
    model.print_trainable_parameters() # 检查参数量
    
    # 3. 优化器 (分层学习率)
    optimizer_grouped_parameters = [
        {'params': model.text_encoder.parameters(), 'lr': Config.LR_TEXT},
        {'params': model.img_encoder.parameters(), 'lr': Config.LR_IMG},
        {'params': model.classifier.parameters(), 'lr': Config.LR_HEAD}
    ]
    optimizer = AdamW(optimizer_grouped_parameters, weight_decay=Config.WEIGHT_DECAY)
    
    # 4. 调度器 & Loss
    total_steps = int(len(train_loader) * Config.EPOCHS / Config.ACCUMULATION_STEPS)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )
    criterion = FocalLoss()
    scaler = torch.cuda.amp.GradScaler() # 混合精度
    
    # 5. 训练状态记录
    history = {'train_loss': [], 'train_acc': [], 'train_f1': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    best_val_f1 = 0
    patience_counter = 0
    
    print(f"\n开始训练! Epochs: {Config.EPOCHS}, Batch: {Config.BATCH_SIZE}, Accum: {Config.ACCUMULATION_STEPS}")
    
    for epoch in range(Config.EPOCHS):
        start_time = time.time()
        model.train()
        
        epoch_loss = 0
        train_preds = []
        train_labels = []
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}")
        
        # --- Batch Loop ---
        for step, batch in enumerate(pbar):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image = batch['image'].to(device)
            img_mask = batch['img_mask'].to(device)
            labels = batch['label'].to(device)
            
            # AMP 前向
            with torch.cuda.amp.autocast():
                logits = model(input_ids, attention_mask, image, img_mask)
                loss = criterion(logits, labels)
                loss = loss / Config.ACCUMULATION_STEPS # 梯度累积归一化
            
            # 反向传播
            scaler.scale(loss).backward()
            
            if (step + 1) % Config.ACCUMULATION_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
            
            epoch_loss += loss.item() * Config.ACCUMULATION_STEPS
            
            # 记录用于计算 Train Metric
            preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
            train_preds.extend(preds)
            train_labels.extend(labels.cpu().numpy())
            
            # 更新进度条 Loss
            pbar.set_postfix({'loss': loss.item() * Config.ACCUMULATION_STEPS})
            
        # --- Epoch End ---
        avg_train_loss = epoch_loss / len(train_loader)
        train_acc = accuracy_score(train_labels, train_preds)
        train_f1 = f1_score(train_labels, train_preds, average='macro')
        
        # Validation
        val_loss, val_acc, val_f1 = eval_fn(val_loader, model, criterion, device)
        
        elapsed = time.time() - start_time
        print(f"Epoch {epoch+1} - Time: {elapsed:.0f}s")
        print(f"  Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
        print(f"  Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
        
        # 记录 History
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)
        
        # Checkpoint & Early Stopping
        if val_f1 > best_val_f1:
            print(f"  >>> Best F1 Improved ({best_val_f1:.4f} -> {val_f1:.4f}). Saving Model...")
            best_val_f1 = val_f1
            save_checkpoint(model, 'best_multimodal_model.pth', DRIVE_BASE_PATH)
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"  >>> No improvement. Patience: {patience_counter}/{Config.PATIENCE}")
            
        if patience_counter >= Config.PATIENCE:
            print("Early Stopping Triggered!")
            break
            
    return history, model

## 7. 开始执行 (Start)
**目标:** 实例化模型并启动训练。
* **前置检查:** 确保图片目录非空，防止空跑。
* **输出:** 返回训练历史记录 (`history`) 和最佳模型对象。

In [ ]:
# ==========================================
# 7. 开始执行 (Start)
# ==========================================

# 确保图片目录不为空，否则报错
if len(os.listdir(Config.IMAGE_DIR)) == 0:
    print("错误: 图片目录为空！请检查第一步的数据搬运是否成功。")
else:
    history, model = run_training()
    print("训练流程结束。")

## 8. 绘制学习曲线 (Learning Curves)
**目标:** 可视化训练过程中的指标变化。
* **诊断:** 通过对比 Train 和 Val 的曲线 (Loss, Acc, F1)，判断模型是否过拟合 (Overfitting) 或欠拟合 (Underfitting)。

In [ ]:
# ==========================================
# 8. 绘制学习曲线 (Learning Curves)
# ==========================================

import matplotlib.pyplot as plt

def plot_history(history):
    epochs = range(1, len(history['train_loss']) + 1)
    
    # 创建 1行3列 的画布
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Loss Curve
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss')
    axes[0].plot(epochs, history['val_loss'], 'r-o', label='Val Loss')
    axes[0].set_title('Loss Curve')
    axes[0].set_xlabel('Epochs')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # 2. Accuracy Curve
    axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train Acc')
    axes[1].plot(epochs, history['val_acc'], 'r-o', label='Val Acc')
    axes[1].set_title('Accuracy Curve')
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    # 3. F1-Macro Curve
    axes[2].plot(epochs, history['train_f1'], 'b-o', label='Train F1')
    axes[2].plot(epochs, history['val_f1'], 'r-o', label='Val F1')
    axes[2].set_title('F1-Macro Curve')
    axes[2].set_xlabel('Epochs')
    axes[2].set_ylabel('F1 Score')
    axes[2].legend()
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()

if 'history' in locals():
    plot_history(history)

## 9. 测试集评估 (Test Set Evaluation)
**目标:** 评估模型的真实泛化能力。
* **独立性:** 使用从未参与训练的 **Test Set** 进行推理。
* **多图投票 (推理阶段):** (可选) 对包含多张图片的用户评论，综合所有图片的预测结果。
* **混淆矩阵:** 直观展示模型在哪些类别上容易混淆 (例如: 容易将 4 星误判为 5 星)。

In [ ]:
# ==========================================
# 9. 测试集评估 (Test Set Evaluation)
# ==========================================

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

def test_model(test_path, model_path):
    print("正在加载测试集...")
    tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
    test_ds = GoogleReviewDataset(Config.TRAIN_PATH.replace('Train', 'Test'), Config.IMAGE_DIR, tokenizer, Config.MAX_LEN, is_train=False)
    test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=4)
    
    print(f"正在加载最佳模型权重: {model_path}...")
    # 重新初始化一个干净的模型结构
    model = MultimodalClassifier(num_classes=Config.NUM_CLASSES)
    # 加载权重
    model.load_state_dict(torch.load(model_path))
    model.to(device)
    model.eval()
    
    print("开始测试集推理...")
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image = batch['image'].to(device)
            img_mask = batch['img_mask'].to(device)
            labels = batch['label'].to(device)
            
            with torch.cuda.amp.autocast():
                logits = model(input_ids, attention_mask, image, img_mask)
            
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            
    # --- 1. 打印核心指标 ---
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    print(f"\n>>> Final Test Results <<<")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-Macro: {f1:.4f}")
    
    # --- 2. 详细分类报告 ---
    # 将索引 0-4 映射回 1-5 星
    target_names = [f'{i} Stars' for i in range(1, 6)]
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=target_names))
    
    # --- 3. 绘制混淆矩阵 ---
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix (Test Set)')
    plt.show()

# 执行测试
# 注意：我们使用 best_multimodal_model.pth，它应该已经在当前目录下
if os.path.exists('best_multimodal_model.pth'):
    test_model(Config.VAL_PATH.replace('Val', 'Test'), 'best_multimodal_model.pth')
else:
    print("未找到最佳模型文件，请检查训练是否成功。")

## 10. 模型备份与下载 (Backup)
**目标:** 确保训练成果的安全交付。
* **双重保险:** 除了训练过程中的自动备份外，提供手动下载 `best_multimodal_model.pth` 的接口，防止 Colab 会话断开导致文件丢失。

In [ ]:
# ==========================================
# 10. 模型备份与下载 (Backup)
# ==========================================

from google.colab import files

print("正在检查模型文件...")

if os.path.exists('best_multimodal_model.pth'):
    print("模型文件大小:")
    !ls -lh best_multimodal_model.pth
    
    # 选项 A: 再次强制复制到 Drive (防止训练中途网络抖动没拷进去)
    drive_save_path = os.path.join(DRIVE_BASE_PATH, 'best_multimodal_model.pth')
    shutil.copy('best_multimodal_model.pth', drive_save_path)
    print(f"已备份至 Google Drive: {drive_save_path}")
    
    # 选项 B: 下载到本地电脑 (可选，文件可能很大，约1-2GB)
    print("准备下载到本地浏览器...")
    files.download('best_multimodal_model.pth')
else:
    print("错误: 未找到模型文件！")